In [42]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [ ]:
date = datetime.today()
today = date.strftime('%Y-%m-%d')
today

In [43]:
db = MySQLDatabase("financialmarkets")

## Se obtienen simbolos de una url

In [44]:
url = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqlisted.txt"

In [ ]:
# Descargar el contenido del archivo
response = requests.get(url)
response.raise_for_status()  # Verificar que la descarga fue exitosa
# Dividir el contenido en una lista de símbolos
data_lines = response.text.strip().split('\n')
# Separar el header
header = data_lines[0].strip().split('|')
# Separar cada línea por '|' y eliminar '\r'
rows = [line.strip().split('|') for line in data_lines[1:]]
# Crear DataFrame
df = pd.DataFrame(rows, columns=header)
print(df.shape)


(5146, 8)


**Aramamos la base**

In [ ]:
# generamos variables no existentes
df['GICS Sector'] = ['SD' for x in df['Symbol']]
df['GICS Sub-Industry'] = ['SD' for x in df['Symbol']]
df['Security'] = df['Security Name']
df['Headquarters Location'] = ['SD' for x in df['Symbol']]
df['CIK'] = ['SD' for x in df['Symbol']]
df['Founded'] = [9999 for x in df['Symbol']]
df['Date added'] = ['0000-00-00' for x in df['Symbol']]
df = df[['Symbol','Security','GICS Sector','GICS Sub-Industry','Headquarters Location', 'Date added','CIK','Founded']]
df.head()


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,SD,SD,SD,0000-00-00,SD,9999
1,AACBR,Artius II Acquisition Inc. - Rights,SD,SD,SD,0000-00-00,SD,9999
2,AACBU,Artius II Acquisition Inc. - Units,SD,SD,SD,0000-00-00,SD,9999
3,AACG,ATA Creativity Global - American Depositary Sh...,SD,SD,SD,0000-00-00,SD,9999
4,AACI,Armada Acquisition Corp. II - Class A Ordinary...,SD,SD,SD,0000-00-00,SD,9999


**Cargamos mercado**

In [47]:
# ---------------------------
# 3️ Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['NASDAQ'],
    'country': ['USA'],
    'currency': ['USD']
})
markets

,market_name,country,currency
0,NASDAQ,USA,USD


In [48]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [57]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD


**Cargamos compañias**

In [58]:
market_id = 1

In [59]:
# Agregar sector_id
#df_sector = df.merge(sector_map, left_on=['GICS Sector','GICS Sub-Industry'], right_on=['sector_name','sub_industry'], how='left')
df.loc[:,"market_id"] = [market_id for x in df['Symbol']]
companies = df[["market_id",'Symbol','Security','GICS Sector','GICS Sub-Industry','Date added','Headquarters Location','CIK','Founded']]
companies.columns = ["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']
companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,1,AACB,Artius II Acquisition Inc. - Class A Ordinary ...,SD,SD,0000-00-00,SD,SD,9999
1,1,AACBR,Artius II Acquisition Inc. - Rights,SD,SD,0000-00-00,SD,SD,9999
2,1,AACBU,Artius II Acquisition Inc. - Units,SD,SD,0000-00-00,SD,SD,9999
3,1,AACG,ATA Creativity Global - American Depositary Sh...,SD,SD,0000-00-00,SD,SD,9999
4,1,AACI,Armada Acquisition Corp. II - Class A Ordinary...,SD,SD,0000-00-00,SD,SD,9999


In [60]:
db.insert_to_db(companies, tabla="companies", batch_size=1000, ignore_duplicates=True)

In [61]:
# Obtener mapping symbol -> company_id
company_map = db.execute_query("SELECT company_id, symbol FROM companies")
company_map

,company_id,symbol
0,1,AACB
1,2,AACBR
2,3,AACBU
3,4,AACG
4,5,AACI
...,...,...
5141,5142,ZYBT
5142,5143,ZYME
5143,5144,ZYN
5144,5145,ZYXI


In [62]:
db.close()

🔒 Conexión cerrada
